In [ ]:
import sys
sys.path.append('/work/SWOT/swotdev/swotdev/stephag/floodplain_dem/scripts')
sys.path.append('/work/SWOT/swotdev/swotdev/stephag/floodplain_dem/src')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import skimage
from skimage.morphology import erosion, dilation

In [ ]:
import mrf_waterland_toolbox as toolbox
from mrf_method import (get_grid_lat_lon_ref2, extract_params_from_pixc, 
                        plot_var_and_zoom, apply_mrf_method, get_labels, get_mean_dem,
                        write_mrf_fpdem_file)

In [ ]:
# Path to the PIXC list
path_pixc_list = ["/work/EXPERT_CENTER/mwec/workspace/HR/science_orbit_sites_public/Bijagos/pixc"]

In [ ]:
# Choose to use a referential DEM file to get the grid or
# 
refdem_grid = False

# Optional if refdem_grid = True
refdem_file = "/work/EXPERT_CENTER/mwec/data/aux_files/sad/production/RefDEM/SWOT_RefDEM_20000101T000000_21000101T000000_20220301T025755_v101/Nom/197/SWOT_RefDEM_Nom_197_174L_20000101T000000_21000101T000000_20220301T025755_v101.nc"

# Optional if refdem_grid = False
extrema = [-16.04, -15.35, 11.15, 11.55]
margin = [0., 0.]
resolution = 0.000277777

# Get a PIXC path and filename
for path in path_pixc_list:
    for root, dire, files in os.walk(path):
        for file in files:
            if file.startswith("SWOT_L2_HR_PIXC_") & file.endswith(".nc"):
                pixc = os.path.join(root, file)

ref_dem, latitude_ref_dem, longitude_ref_dem = get_grid_lat_lon_ref2(refdem_grid, 
                                                                     path_ref_dem=refdem_file, 
                                                                     lonlat_extrema=extrema,
                                                                     resolution=resolution)
print(ref_dem)

In [ ]:
plot = True
# Select a small part of the region (range/azimuth indexes) for visualization
l0, l1 = 300, 650
c0, c1 = 500, 1200
zoom = (slice(l0, l1), slice(c0, c1))
min_cross_track = 20000

fpdem = []
for path in path_pixc_list:
    for root, dire, files in os.walk(path):
        for file in files:
            if file.startswith("SWOT_L2_HR_PIXC_") & file.endswith(".nc"):
                print(path, file)
                
                (grid_h, grid_h_interp, 
                 grid_inc, 
                 grid_sig0, grid_sig0_interp, 
                 grid_coh_interp, 
                 mask, 
                 date, 
                 grid_coh_th_interp) = extract_params_from_pixc(file, root, ref_dem, min_crosstrack=min_cross_track)

                if plot:
                    fig, axes = plt.subplots(1, 2, figsize=(10, 5)) # 1 ligne, 2 colonnes
                    im1 = axes[0].imshow(10*np.log10(grid_sig0)[zoom], cmap="Greys_r")
                    im2 = axes[1].imshow(grid_h[zoom], vmin=1, vmax=5)
                    axes[1].set_title(f'Height - {date}')
                    axes[0].set_title(f'Sig0 (dB) - {date}')

                fpdem.append([grid_h, grid_h_interp, grid_inc, grid_sig0, grid_sig0_interp, 
                              grid_coh_interp, mask, date, grid_coh_th_interp])

In [ ]:
# Sorted by date
fpdem = sorted(fpdem, key=lambda fpdem: fpdem[7])
print('All dates: ', [row[7] for row in fpdem])

In [ ]:
# Chosen date for unitary test
ind_date = 4

In [ ]:
# Plot sig0 before and after erosion/dilation for the first date, on the whole region
tab_tmp = np.where(fpdem[ind_date][0] == 0., 0, 1)
kernel = np.ones([5,5])
tab_tmp2 = erosion(dilation(tab_tmp, kernel), kernel)

fig, axes = plt.subplots(1, 2, figsize=(10, 5)) # 1 ligne, 2 colonnes
axes[0].imshow(tab_tmp)
axes[1].imshow(tab_tmp2)

In [ ]:
# Unit test on one date

# ------------------------------------------
# 4. PROCESSING (MRF)
# ------------------------------------------

pixel_res_m = 30.0
tile_size_km = 25
stride_km = 20.0
border_exclude_km = 2.5

land_law = 'gaussian'  # Options: 'gaussian' or 'exponnorm'
weight_ssh = 0.5
max_iters = 50
convergence = 0.02

p_sig0_init = {
    'mu_land': 7.0, 'std_land': 3.5,
    'mu_water': 10.0, 'std_water': 2.5
}

p_ssh_init = {
    'mu_land': 0.25, 'std_land': 1.,
    'mu_water': 0.0, 'std_water': 0.1,
    'expon_k': 1.0, 'expon_loc': 0.1, 'expon_scale': 0.05
}

(height_cycle_0, sig0_cycle_0,
 proba_smooth_0, labels_smooth_0, 
 proba_smooth_h, labels_smooth_h) = apply_mrf_method(
    fpdem[ind_date],
    p_sig0_init, p_ssh_init, land_law, max_iters, convergence, weight_ssh,
    pixel_res_m, tile_size_km, stride_km, border_exclude_km
)

proba_map_list = [[proba_smooth_0, labels_smooth_0, proba_smooth_h, labels_smooth_h]]

In [ ]:
# Visualization of the unit test, on the whole region and zoomed part
 
# sig0
plot_var_and_zoom(sig0_cycle_0, "sig0_cycle_0", cmap="Greys_r", zoom=zoom)
 
# Height
plot_var_and_zoom(height_cycle_0, "h_cycle_0", vmin=-2, vmax=2, zoom=zoom)

# Proba_smooth
plot_var_and_zoom(np.where(proba_smooth_0 > 0, proba_smooth_0, 1), "proba_smooth_0", zoom=zoom)
plot_var_and_zoom(np.where(proba_smooth_h > 0, proba_smooth_h, 1), "proba_smooth_h", zoom=zoom)

# Label_smooth
plot_var_and_zoom(np.where(labels_smooth_0 > -100, labels_smooth_0, -1), "labels_smooth_0", zoom=zoom)
plot_var_and_zoom(np.where(labels_smooth_h > -100, labels_smooth_h, -1), "labels_smooth_h", zoom=zoom)

#
fig, axes = plt.subplots(1, 3, figsize=(30, 10))  # 1 ligne, 2 colonnes

axes[0].set_title("swot sig0 (dB) " + fpdem[ind_date][7])
axes[1].set_title("proba map without height " + fpdem[ind_date][7])
axes[2].set_title("proba map with height " + fpdem[ind_date][7])

cax0 = make_axes_locatable(axes[0]).append_axes("right", size="5%", pad=0.05)
cax1 = make_axes_locatable(axes[1]).append_axes("right", size="5%", pad=0.05)
cax2 = make_axes_locatable(axes[2]).append_axes("right", size="5%", pad=0.05)

sig0_plot = axes[0].imshow(10*np.log10(fpdem[ind_date][3])[zoom], vmin=0, vmax=15, cmap="Greys_r")
proba_plot = axes[1].imshow(proba_smooth_0[zoom])
proba_plot_with_height = axes[2].imshow(proba_smooth_h[zoom])

fig.colorbar(sig0_plot, ax=axes[0], cax=cax0)
fig.colorbar(proba_plot, ax=axes[1], cax=cax1)
fig.colorbar(proba_plot_with_height, ax=axes[2], cax=cax2)

In [ ]:
# Get the combined labels and show the results on the zoomed region
threshold_without_height = 0.2
threshold_with_height = 0.2
labels_combined = get_labels(fpdem, 0, proba_map_list, threshold_with_height, threshold_without_height)

# Plot, zoomed on a small part of the region
fig, axes = plt.subplots(1, 3, figsize=(30, 10))  # 1 row, 2 columns

axes[0].set_title("swot dem " + fpdem[0][7])
axes[1].set_title("swot sig0 (dB) " + fpdem[0][7])
axes[2].set_title("label map combined " + fpdem[0][7])

cax0 = make_axes_locatable(axes[0]).append_axes("right", size="5%", pad=0.05)
cax1 = make_axes_locatable(axes[1]).append_axes("right", size="5%", pad=0.05)
cax2 = make_axes_locatable(axes[2]).append_axes("right", size="5%", pad=0.05)

dem_plot = axes[0].imshow(fpdem[0][0].data[zoom], vmin=0, vmax=5)
sig0_plot = axes[1].imshow(10 * np.log10(fpdem[0][3])[zoom], vmin=0, vmax=15, cmap="Greys_r")
labels_plot_combined = axes[2].imshow(labels_combined[zoom])

fig.colorbar(dem_plot, ax=axes[0], cax=cax0)
fig.colorbar(sig0_plot, ax=axes[1], cax=cax1)
fig.colorbar(labels_plot_combined, ax=axes[2], cax=cax2)